In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Pin to a single GPU. Unsloth's free/open-source
                                           # version trains on one GPU — this keeps it well
                                           # behaved even if you picked "GPU T4 x2".

!nvidia-smi

Wed Sep  9 05:24:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
# %%capture hides the (very long) pip install logs. If something goes wrong below,
# delete the line above and re-run this cell so you can see the actual error.

!pip install --upgrade pip
!pip install unsloth

# If this cell fails with a dependency conflict (Kaggle's base image changes over time),
# uncomment the next line and re-run:
# !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [3]:
BASE_MODEL     = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # Unsloth's pre-quantized 4-bit repo
MAX_SEQ_LENGTH = 2048            # data-science Q&A / code rarely needs more; raise if yours is longer
LOAD_IN_4BIT   = True            # QLoRA: lowest VRAM, fastest iteration

DATASET_NAME   = "ed001/ds-coder-instruct-v1"  # Alpaca-style data-science / ML coding instructions, ~16.4k rows
                                                # swap for your own dataset — see the formatting cell below

OUTPUT_DIR     = "/kaggle/working/checkpoints"
FINAL_LORA_DIR = "/kaggle/working/qwen25_3b_datascience_lora"

In [4]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # auto: bf16 on Ampere+, fp16 on older GPUs (P100/T4)
    load_in_4bit   = LOAD_IN_4BIT,
)

# Qwen2.5's ChatML template: <|im_start|>role ... <|im_end|>
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,    # LoRA rank — 8-16 is plenty for domain adaptation
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 16,
    lora_dropout               = 0,     # 0 = Unsloth's fused fast path
    bias                       = "none",# "none" = Unsloth's fused fast path
    use_gradient_checkpointing = "unsloth",  # less VRAM than vanilla checkpointing
    random_state                = 3407,
    use_rslora                  = False,
    loftq_config                 = None,
)

Unsloth 2026.9.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

raw_dataset = load_dataset(DATASET_NAME, split="train")

def to_conversation(example):
    user_msg = example["instruction"]
    if example.get("input"):
        user_msg += f"\n\n{example['input']}"
    return {
        "conversations": [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": example["output"]},
        ]
    }

dataset = raw_dataset.map(to_conversation, remove_columns=raw_dataset.column_names)
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

dataset = dataset.train_test_split(test_size=0.02, seed=3407)
train_dataset, eval_dataset = dataset["train"], dataset["test"]

print(f"Train examples: {len(train_dataset)} | Eval examples: {len(eval_dataset)}")
print("\nSample formatted training example:\n" + "-" * 60)
print(train_dataset[0]["text"][:800])

README.md: 0.00B [00:00, ?B/s]

ds_coder.jsonl:   0%|          | 0.00/39.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16409 [00:00<?, ? examples/s]

Map:   0%|          | 0/16409 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/16409 [00:00<?, ? examples/s]

Map:   0%|          | 0/16409 [00:00<?, ? examples/s]

Train examples: 16080 | Eval examples: 329

Sample formatted training example:
------------------------------------------------------------
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Use TensorFlow to build a deep learning model for predicting the future price of a given stock using a time series dataset with multiple features, such as volume, moving averages, and technical indicators. Additionally, incorporate a custom loss function that penalizes false negatives more than false positives in order to optimize the model for risk management.<|im_end|>
<|im_start|>assistant
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam

# Load the time series dataset with multiple features
# Assuming the dataset is loaded into X_train and y_train

# 


In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length               = MAX_SEQ_LENGTH,
        per_device_train_batch_size  = 2,   # P100/T4 have 16GB — raise to 4 for extra speed if VRAM allows
        gradient_accumulation_steps  = 8,   # effective batch size = 2 * 8 = 16
        per_device_eval_batch_size   = 2,
        warmup_steps                 = 20,
        num_train_epochs             = 3,   # or comment out and set max_steps=60 below for a fast smoke test first
        # max_steps                  = 60,
        learning_rate                 = 2e-4,
        logging_steps                 = 10,
        eval_strategy                 = "steps",
        eval_steps                    = 100,
        save_strategy                 = "steps",
        save_steps                    = 200,
        save_total_limit              = 2,
        optim                         = "adamw_8bit",
        weight_decay                   = 0.01,
        lr_scheduler_type               = "cosine",
        seed                             = 3407,
        output_dir                       = OUTPUT_DIR,
        report_to                        = "none",
        packing                          = True,   # concatenate short sequences -> no wasted padding compute
        dataset_num_proc                 = 2,
    ),
)

# Mask the loss on prompt tokens — the model only learns from its own responses.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/16080 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/16080 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/329 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/329 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


Map:   0%|          | 0/3667 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3667 [00:00<?, ? examples/s]

Unsloth: Removed 3 out of 3667 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map:   0%|          | 0/77 [00:00<?, ? examples/s]

In [8]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = torch.cuda.max_memory_reserved() / 1024**3
print(f"GPU = {gpu_stats.name} | Total VRAM = {gpu_stats.total_memory / 1024**3:.1f} GB | Reserved before training = {start_mem:.2f} GB")

trainer_stats = trainer.train()

used_mem = torch.cuda.max_memory_reserved() / 1024**3;
print(f"\nTraining took {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")
print(f"Peak reserved VRAM during training: {used_mem:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU = Tesla T4 | Total VRAM = 14.6 GB | Reserved before training = 2.09 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,664 | Num Epochs = 3 | Total steps = 687
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.604847,0.625557
200,0.601146,0.620034
300,0.577125,0.618893
400,0.586129,0.616818
500,0.559589,0.618425
600,0.563074,0.618715
687,0.580744,0.618497


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-687/tokenizer_config.json.



Training took 442.8 minutes
Peak reserved VRAM during training: 6.51 GB


In [9]:
model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)
print(f"LoRA adapter saved to {FINAL_LORA_DIR}")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25_3b_datascience_lora/tokenizer_config.json.


LoRA adapter saved to /kaggle/working/qwen25_3b_datascience_lora


In [10]:
FastLanguageModel.for_inference(model)  # Unsloth's ~2x faster inference path

TEST_PROMPTS = [
    "Write a Python function to compute the F1 score from a confusion matrix.",
    "Explain the bias-variance tradeoff and how it shows up when tuning a random forest.",
    "I have a pandas DataFrame with missing values in a numeric column. Show two ways to handle it.",
]

for prompt in TEST_PROMPTS:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    print("\n" + "=" * 80)
    print(f"PROMPT: {prompt}")
    print("=" * 80)

    output_ids = model.generate(
        input_ids=inputs, max_new_tokens=400, temperature=0.7, top_p=0.9, do_sample=True,
    )
    print(tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: Write a Python function to compute the F1 score from a confusion matrix.


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


import numpy as np

def f1_score(cm):
    precision = cm[0][0] / (cm[0][0] + cm[0][1])
    recall = cm[0][0] / (cm[0][0] + cm[1][0])
    return 2 * ((precision * recall) / (precision + recall))

PROMPT: Explain the bias-variance tradeoff and how it shows up when tuning a random forest.


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The bias-variance tradeoff is a concept in machine learning that refers to the balance between two sources of error: bias and variance. Bias is the error due to modeling assumptions in a learning algorithm that prevent it from fitting the data correctly. Variance is the error from sensitivities to small fluctuations in the training set.

When tuning a random forest, we often want to minimize both bias and variance. If the model has high bias, it will underfit the training data, resulting in poor performance on both the training and test sets. On the other hand, if the model has high variance, it will overfit the training data, resulting in poor performance on the test set but good performance on the training set. 

Random forests are typically used for their ability to handle large datasets and high-dimensional spaces. They do this by creating multiple decision trees and averaging their predictions, which helps to reduce variance and improve generalization. However, if too many trees a

In [15]:
# ============================================================
# FIND YOUR FINE-TUNED MODEL AUTOMATICALLY
# ============================================================

import os

print("=" * 80)
print("SEARCHING /kaggle/working FOR FINE-TUNED MODEL")
print("=" * 80)

# Candidate directories from your training notebook
candidates = [
    "/kaggle/working/merged_16bit",
    "/kaggle/working/qwen25-datascience-merged",
    "/kaggle/working/qwen25-datascience-merged-4bit",
    "/kaggle/working/fine_tuned_model",
    "/kaggle/working/qwen25-datascience-lora",
    "/kaggle/working/qwen25_3b_datascience_lora",
    "/kaggle/working/checkpoints/checkpoint-687",
    "/kaggle/working/checkpoints/checkpoint-600",
    "/kaggle/working/checkpoints/checkpoint-400",
    "/kaggle/working/checkpoints/checkpoint-200",
]

found = []

for path in candidates:

    if os.path.isdir(path):

        files = os.listdir(path)

        # Look for actual model/tokenizer artifacts
        has_model = any(
            f.endswith((
                ".safetensors",
                ".bin",
                ".pt",
                ".pth"
            ))
            for f in files
        )

        has_config = (
            "config.json" in files
            or "adapter_config.json" in files
        )

        if has_model or has_config:
            found.append(path)

# Also recursively search if none of the expected paths exist
if not found:

    print("\nNo standard directory found.")
    print("Searching recursively...\n")

    for root, dirs, files in os.walk(
        "/kaggle/working"
    ):

        relevant = any(
            f.endswith((
                ".safetensors",
                ".bin",
                ".pt",
                ".pth"
            ))
            for f in files
        )

        if relevant:

            found.append(root)

# Remove duplicates
found = list(dict.fromkeys(found))

if not found:

    print(
        "❌ NO MODEL ARTIFACT FOUND."
    )

    print(
        "\nYour training completed, but the model "
        "was probably not exported/saved."
    )

else:

    print(
        f"✅ Found {len(found)} possible model directories:\n"
    )

    for i, path in enumerate(
        found,
        start=1
    ):

        print(f"{i}. {path}")

        try:
            size_gb = (
                sum(
                    os.path.getsize(
                        os.path.join(path, f)
                    )
                    for f in os.listdir(path)
                    if os.path.isfile(
                        os.path.join(path, f)
                    )
                )
                / (1024 ** 3)
            )

            print(
                f"   Size: {size_gb:.2f} GB"
            )

        except:
            pass

# ------------------------------------------------------------
# Choose the best available candidate automatically
# ------------------------------------------------------------

priority = [
    "/kaggle/working/merged_16bit",
    "/kaggle/working/qwen25-datascience-merged",
    "/kaggle/working/fine_tuned_model",
    "/kaggle/working/checkpoints/checkpoint-687",
    "/kaggle/working/checkpoints/checkpoint-600",
    "/kaggle/working/checkpoints/checkpoint-400",
    "/kaggle/working/checkpoints/checkpoint-200",
    "/kaggle/working/qwen25-datascience-lora",
    "/kaggle/working/qwen25_3b_datascience_lora",
]

FINE_TUNED_MODEL = None

for path in priority:

    if path in found:

        FINE_TUNED_MODEL = path
        break

# If priority paths weren't found, use the first discovered directory.
if FINE_TUNED_MODEL is None and found:

    FINE_TUNED_MODEL = found[0]

print("\n" + "=" * 80)

if FINE_TUNED_MODEL:

    print(
        "✅ SELECTED FINE-TUNED MODEL:"
    )

    print(
        FINE_TUNED_MODEL
    )

    print("=" * 80)

else:

    print(
        "❌ NO FINE-TUNED MODEL COULD BE LOCATED."
    )

    print("=" * 80)

SEARCHING /kaggle/working FOR FINE-TUNED MODEL
✅ Found 3 possible model directories:

1. /kaggle/working/qwen25_3b_datascience_lora
   Size: 0.12 GB
2. /kaggle/working/checkpoints/checkpoint-687
   Size: 0.18 GB
3. /kaggle/working/checkpoints/checkpoint-600
   Size: 0.18 GB

✅ SELECTED FINE-TUNED MODEL:
/kaggle/working/checkpoints/checkpoint-687


In [22]:
import shutil
import os

src = "/kaggle/working/qwen25_3b_datascience_lora"
dst = "/kaggle/working/qwen25_3b_datascience_lora.zip"

if not os.path.isdir(src):
    raise FileNotFoundError(f"Not found: {src}")

shutil.make_archive(
    "/kaggle/working/qwen25_3b_datascience_lora",
    "zip",
    src
)

print("Created:")
print(dst)
print("Size:", round(os.path.getsize(dst) / 1024**2, 2), "MB")

Created:
/kaggle/working/qwen25_3b_datascience_lora.zip
Size: 107.91 MB
